# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### RunnableLamda

일반 Python 함수를 LCEL 체인에서 사용할 수 있는 Runnable 형태로 wrapping 처리해주는 클래스

In [ ]:
# 입력을 받아 내장된 함수를 실행하는 Runnable
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕 만나서 반갑다~')

11

In [ ]:
# batch() : 여러 건의 입력을 일괄처리해줌
runnable.batch(['안녕 만나서 반갑다!', '너도? 나도', '?!', '😊😊😊😊'])

[11, 6, 2, 4]

In [5]:
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

celsius_temps = [0, 25, 100, -10, 37]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temps)


[32.0, 77.0, 212.0, 14.0, 98.6]

In [ ]:
import time # 출력 딜레이용

def generator(x):
    for y in x:     # 입력을 문자 단위로 순회
        yield y     # 한 글자씩 반환 (스트리밍 방식) 

runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세요~😊😊😊😊안녕하세요~😊😊😊😊안녕하세요~😊😊😊😊안녕하세요~😊😊😊😊'):
    print(chunk, end='', flush=True) # chunk를 줄바꿈없이 즉시 출력
    time.sleep(0.1) # 글자 출력마다 딜레이 0.1초

안녕하세요~😊😊😊😊안녕하세요~😊😊😊😊안녕하세요~😊😊😊😊안녕하세요~😊😊😊😊

In [7]:
# 사용 예시
def gen(x):
    for y in x:
        yield y

gen10 = gen(range(10))

for n in gen10:
    print(n)
    

0
1
2
3
4
5
6
7
8
9


In [ ]:
# next(gen10) # 제너레이터 다음 값 1개 반환(다 꺼내고 나면 StopIteration 발생)

StopIteration: 

### RunnableSequence

Runnable 객체를 순차연결해주는 Runnable 객체

In [ ]:
from langchain_core.runnables import RunnableSequence

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableSequence(runnable1, runnable2) # runnable1 -> runnable2
chain.invoke(3)


[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [10]:
chain = runnable1 | runnable2
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [11]:
chain = runnable2 | runnable1
chain.invoke(3)

{'foo': [3, 3, 3]}

### RunnableParallel

여러 Runnable 객체를 인자로 받아, 병렬처리 후 각각의 응답을 하나의 dict로 반환

In [13]:
# 여러 Runnable 들을 같은 입력으로 병렬로 실행
from langchain_core.runnables import RunnableParallel 

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableParallel(r1 = runnable1, r2 = runnable2) # r1, r2 를 병렬 실행해 dict 로 반환
chain.invoke(3)

{'r1': {'foo': 3}, 'r2': [3, 3, 3]}

- 사용자가 준 주제를 이용해 삼행시, 농담, 시를 각각 생성해서 하나의 응답으로 반환

In [ ]:
from langchain_core.prompts import PromptTemplate   # prompt 구성
from langchain.chat_models import init_chat_model   # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser # 답변 문자형 변환
# 여러 Runnable 들을 같은 입력으로 병렬로 실행
from langchain_core.runnables import RunnableParallel 

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시 지어주세요. 주제 : {topic}'
)

n_poem_chain = acrostic_poem_prompt | llm | output_parser

joke_prompt = PromptTemplate.from_template(
    '당신은 한국식 농담계의 엄청난 고수입니다. 다음 주제로 배꼽이 빠질만한 농담을 지어주세요. 주제 : {topic}'
)

joke_chain = joke_prompt | llm | output_parser

poem_prompt = PromptTemplate.from_template(
    '당신은 엄청난 현대시 작가입니다. 다음 주제로 눈물이 나올 정도의 감성적인 시를 지어주세요. 주제 : {topic}'
)

poem_chain = poem_prompt | llm | output_parser

# 동일 입력(topic)으로 3개 체인을 병렬 실행 {acrostic_poem: 실행결과, ...}
chain = RunnableParallel(
    acrostic_poem = n_poem_chain,
    joke = joke_chain,
    poem = poem_chain
)

def combine_result(input_dict: dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem = input_dict['poem']
    return f"""
    n행시 :
    {acrostic_poem}

    농담 :
    {joke}

    현대시:
    {poem}
    """

chain = chain | RunnableLambda(combine_result)
print(chain.invoke({'topic': '동섭이'}))



    n행시 :
    동: 동네방네 자랑하고 싶은 매력에  
섭: 섭외 1순위로 불리는 존재감까지 갖춘  
이: 이 세상 하나뿐인 멋진 동섭이!

    농담 :
    동섭이가 PC방에 갔는데 사장님이 갑자기 소리쳤어요.

“동접 100명 넘었습니다!”

그러자 동섭이가 벌떡 일어나며 말했대요.

“저요? 전 혼자 왔는데요!” 😄

    현대시:
    ### 동섭이

동섭이는  
늘 한 박자 늦게 웃었다  

사람들이 다 떠난 뒤에야  
문득 생각난 사람처럼  
조용히 웃었다  

그의 주머니에는  
구겨진 버스표 하나와  
끝내 전하지 못한 말들이  
계절처럼 쌓여 있었다  

“괜찮아”라고 말할 때마다  
괜찮지 않은 것들이  
그의 목소리 안에서  
작게 젖어 갔다  

어느 겨울 저녁,  
동섭이는 내게 장갑 한 짝을 건넸다  

자기는 손이 시리지 않다며  
끝까지 웃었지만  
나는 그날 처음 알았다  

사람이란  
추위를 참는 것이 아니라  
누군가의 손을 따뜻하게 하느라  
자기 온기를 다 써버리는 존재라는 걸  

동섭이는 떠날 때도  
아무것도 가져가지 않았다  

다만 창가에  
오래된 햇빛 하나를 남겨 두었다  

그래서 지금도  
해 질 무렵이면  
방 안이 조금 붉어진다  

나는 그 빛을 보며  
동섭아, 하고 불러 본다  

대답은 오지 않지만  
이름 하나가  
가슴 깊은 곳에서 무너지는 소리는  

아직도  
너무 선명하다.
    


### RunnablePassThrough
- 사용자의 입력값을 그대로 전달해주는 Runnable


In [ ]:
acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시 지어주세요. 주제 : {topic}'
)

n_poem_chain = acrostic_poem_prompt | llm | output_parser
print(n_poem_chain.invoke({'topic' : '텀블러'}))

텀: 텀을 두고 천천히 마셔도  
블: 블링블링한 내 텀블러 속 커피는 따뜻하고  
러: 러브처럼 매일 챙기게 되는 나의 필수품!


In [29]:
from langchain_core.runnables import RunnablePassthrough 

prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시 지어주세요. 주제 : {topic}'
)

chain = {'topic': RunnablePassthrough()} | prompt | llm | output_parser
print(n_poem_chain.invoke({'topic' : '학원수업'}))

**학**: 학교 끝나면 또 시작되는 열정의 시간  
**원**: 원하는 꿈을 향해 한 걸음씩 나아가고  
**수**: 수많은 문제도 포기하지 않고 풀다 보면  
**업**: 업그레이드된 실력으로 꿈에 한층 가까워진다!


In [ ]:
prompt = PromptTemplate.from_template("""
당신은 {n}행시의 엄청난 고수입니다. 다음 주제로 {n}행시 지어주세요.

주제 : {topic}

출력형식 :
==== <주제> <n행시> ====
<n행시 작성>
""")

chain = ({'topic': RunnablePassthrough()} 
         | RunnablePassthrough.assign( # 기존 topic만 있던 dict -> 새 key를 추가
            n = lambda x: len(x['topic']), # n = topic 길이
            k = lambda x: 100              # k = 100 (미사용)
        ) 
        | prompt # 확장된 dict(topic, n, k) 를 프롬프트에 주입하여 완성
        | llm 
        | output_parser
)
print(chain.invoke('아이스크림'))

==== 아이스크림 5행시 ====
아: 아삭한 설렘이 입안 가득 퍼지고  
이: 이 순간만큼은 더위도 잠시 멈추고  
스: 스르르 녹아드는 달콤한 행복  
크: 크게 한입 베어 물면 웃음이 피어나며  
림: 림처럼 부드러운 하루가 완성된다!
